# Comparación de modelos — House Pricing

Este notebook reproduce el torneo de modelos y genera los datos del artefacto de comparación. Para cada modelo calcula su desempeño en validación, sus predicciones sobre `test.csv` y una evaluación de coherencia frente a la inferencia. El resultado se guarda en `reports/model_comparison.json`.

No modifica los modelos guardados en `models/`, `pipeline_config.yaml` ni `predictions.csv`.

## 1. Librerías

Importación de las librerías para manipulación de datos, modelado, validación cruzada y análisis estadístico.

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import yaml
from scipy import stats
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import (
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, learning_curve, train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 2. Configuración y carga de datos

Lectura de `pipeline_config.yaml`, de los datos crudos y de las predicciones de la inferencia. La división 80/20 usa los mismos parámetros del entrenamiento.

In [2]:
PROJ_ROOT = Path("..")

with open(PROJ_ROOT / "pipeline_config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

TARGET = config["TARGET"]
MODEL_FEATURES = config["MODEL_FEATURES"]
RANDOM_STATE = config["SPLIT"]["RANDOM_STATE"]
RAW_DIR = PROJ_ROOT / config["DATA"]["RAW_DIR"]

raw_train = pd.read_csv(RAW_DIR / config["DATA"]["TRAIN_FILE"])
raw_test = pd.read_csv(RAW_DIR / config["DATA"]["TEST_FILE"])
inference = pd.read_csv(PROJ_ROOT / config["INFERENCE"]["PREDICTIONS_PATH"])[TARGET].to_numpy()

X_train, X_valid, y_train_real, y_valid_real = train_test_split(
    raw_train[MODEL_FEATURES],
    raw_train[TARGET],
    test_size=config["SPLIT"]["TEST_SIZE"],
    random_state=RANDOM_STATE,
)
y_train = np.log1p(y_train_real)
X_test = raw_test[MODEL_FEATURES]
train_prices = raw_train[TARGET].to_numpy()
target_range = y_train_real.max() - y_train_real.min()

print(f"X_train: {X_train.shape} | X_valid: {X_valid.shape} | X_test: {X_test.shape}")

X_train: (1168, 30) | X_valid: (292, 30) | X_test: (1459, 30)


## 3. Modelos e hiperparámetros

Definición idéntica a la sección 5 de `3.0-gd-entrenamiento-modelos.ipynb`. El preprocesamiento se clona sin ajustar para reajustarse dentro de cada fold, lo que evita la fuga de datos.

In [3]:
N_FOLDS = 10
SCORING = "neg_root_mean_squared_error"
TRAIN_SIZES = np.linspace(0.4, 1.0, 4)

cv = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
preprocessing = clone(joblib.load(PROJ_ROOT / config["PIPELINE_PATH"]))


def build_pipeline(model):
    """Une el preprocesamiento sin ajustar con un modelo."""
    return Pipeline([("preprocessing", clone(preprocessing)), ("model", model)])


models = {
    "Ridge": {
        "model": Ridge(),
        "grid": {"model__alpha": [0.1, 1, 10, 30, 100]},
    },
    "ElasticNet": {
        "model": ElasticNet(max_iter=20_000, random_state=RANDOM_STATE),
        "grid": {
            "model__alpha": [0.0005, 0.001, 0.01],
            "model__l1_ratio": [0.2, 0.5, 0.8],
        },
    },
    "RandomForest": {
        "model": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE),
        "grid": {
            "model__max_depth": [10, None],
            "model__min_samples_leaf": [2, 5],
            "model__max_features": [0.2, 0.33, 0.5],
        },
    },
    "GradientBoosting": {
        "model": GradientBoostingRegressor(n_estimators=500, random_state=RANDOM_STATE),
        "grid": {
            "model__learning_rate": [0.02, 0.05],
            "model__max_depth": [2, 3, 4],
            "model__subsample": [0.7, 1.0],
        },
    },
    "HistGradientBoosting": {
        "model": HistGradientBoostingRegressor(
            max_iter=300, l2_regularization=1.0, random_state=RANDOM_STATE
        ),
        "grid": {
            "model__learning_rate": [0.03, 0.05],
            "model__max_leaf_nodes": [7, 15],
            "model__min_samples_leaf": [10, 20],
        },
    },
    "MLP": {
        "model": TransformedTargetRegressor(
            regressor=MLPRegressor(
                max_iter=2_000,
                early_stopping=True,
                n_iter_no_change=30,
                random_state=RANDOM_STATE,
            ),
            transformer=StandardScaler(),
        ),
        "grid": {
            "model__regressor__hidden_layer_sizes": [(32,), (64,), (64, 32)],
            "model__regressor__alpha": [0.01, 0.1, 1.0],
        },
    },
}

## 4. Reentrenamiento del torneo

Se repite la búsqueda de hiperparámetros con validación cruzada de 10 folds. Los resultados deben coincidir con los del notebook de entrenamiento; la sección 5 lo comprueba con el RMSE del ganador.

In [4]:
searches = {}

for name, spec in models.items():
    search = GridSearchCV(
        build_pipeline(spec["model"]),
        spec["grid"],
        cv=cv,
        scoring=SCORING,
        n_jobs=-1,
        return_train_score=True,
    )
    searches[name] = search.fit(X_train, y_train)
    print(f"{name}: RMSE CV (log) = {-search.best_score_:.4f}")

Ridge: RMSE CV (log) = 0.1290


ElasticNet: RMSE CV (log) = 0.1288


RandomForest: RMSE CV (log) = 0.1338


GradientBoosting: RMSE CV (log) = 0.1220


HistGradientBoosting: RMSE CV (log) = 0.1231


MLP: RMSE CV (log) = 0.1280


## 5. Métricas en validación

Métricas de cada modelo en dólares sobre el conjunto de validación, junto con el error en escala logarítmica de cada etapa. Las brechas son las mismas del torneo. El sesgo es el promedio de real − predicho: un valor positivo indica que el modelo subestima los precios.

In [5]:
valid_predictions = {}
rows = []

for name, search in searches.items():
    pred = np.expm1(search.predict(X_valid))
    valid_predictions[name] = pred
    best = search.best_index_
    rmse_train_log = -search.cv_results_["mean_train_score"][best]
    rmse_cv_log = -search.best_score_
    rmse_valid_log = root_mean_squared_error(np.log1p(y_valid_real), np.log1p(pred))
    rmse_real = root_mean_squared_error(y_valid_real, pred)

    rows.append(
        {
            "modelo": name,
            "rmse_real": rmse_real,
            "rmse_normalizado": rmse_real / target_range,
            "mae": mean_absolute_error(y_valid_real, pred),
            "mape_pct": mean_absolute_percentage_error(y_valid_real, pred) * 100,
            "r2": r2_score(y_valid_real, pred),
            "sesgo": (y_valid_real - pred).mean(),
            "rmse_train_log": rmse_train_log,
            "rmse_cv_log": rmse_cv_log,
            "rmse_valid_log": rmse_valid_log,
            "brecha_train_cv_pct": (rmse_cv_log / rmse_train_log - 1) * 100,
            "brecha_cv_valid_pct": (rmse_valid_log / rmse_cv_log - 1) * 100,
        }
    )

metrics = pd.DataFrame(rows).sort_values("rmse_real").reset_index(drop=True)

assert metrics.loc[0, "modelo"] == config["WINNER"]["NAME"]
assert np.isclose(metrics.loc[0, "rmse_real"], config["WINNER"]["RMSE_REAL"])
metrics.round(4)

,modelo,rmse_real,rmse_normalizado,mae,mape_pct,r2,sesgo,rmse_train_log,rmse_cv_log,rmse_valid_log,brecha_train_cv_pct,brecha_cv_valid_pct
0,GradientBoosting,26686.8387,0.0376,16006.3543,9.4505,0.9072,1403.4775,0.0587,0.1220,0.1367,107.8026,12.0761
1,MLP,27937.3352,0.0393,16694.2971,9.7938,0.8982,1967.0451,0.0971,0.1280,0.1388,31.8771,8.4396
2,RandomForest,28423.9453,0.0400,16267.8260,9.7266,0.8947,3104.9742,0.0663,0.1338,0.1451,101.8107,8.3937
3,HistGradientBoosting,28820.1727,0.0406,16768.8001,9.9231,0.8917,111.0519,0.0562,0.1231,0.1432,118.8853,16.3610
4,Ridge,30437.6863,0.0429,17108.8064,9.5572,0.8792,2670.1796,0.1257,0.1290,0.1333,2.5945,3.3774
5,ElasticNet,30690.4509,0.0432,16964.4868,9.4874,0.8772,2805.6356,0.1258,0.1288,0.1335,2.3796,3.6480


## 6. Predicciones sobre `test.csv`

Cada modelo predice las 1,459 viviendas de `test.csv`. Las predicciones se resumen y se comparan con los precios reales de `train.csv` y con la salida de la inferencia. La prueba de Kolmogórov-Smirnov (KS) mide la distancia entre dos distribuciones. El p-valor se reporta como referencia y no se usa para descartar modelos, porque con muestras de este tamaño detecta diferencias muy pequeñas.

In [6]:
test_predictions = {name: np.expm1(search.predict(X_test)) for name, search in searches.items()}

test_summary = pd.DataFrame(
    [
        {
            "modelo": name,
            "vacios": int(np.isnan(pred).sum()),
            "negativos": int((pred <= 0).sum()),
            "fuera_de_rango": int(((pred < train_prices.min()) | (pred > train_prices.max())).sum()),
            "minimo": pred.min(),
            "maximo": pred.max(),
            "media": pred.mean(),
            "mediana": np.median(pred),
            "desviacion": pred.std(ddof=1),
            "asimetria": stats.skew(pred),
            "ks_train": stats.ks_2samp(pred, train_prices).statistic,
            "ks_train_p": stats.ks_2samp(pred, train_prices).pvalue,
            "ks_inferencia": stats.ks_2samp(pred, inference).statistic,
            "corr_inferencia": np.corrcoef(pred, inference)[0, 1],
            "dif_mediana_inferencia": np.median(np.abs(pred - inference)),
        }
        for name, pred in test_predictions.items()
    ]
).set_index("modelo")

test_summary.round(3)

,vacios,negativos,fuera_de_rango,minimo,maximo,media,mediana,desviacion,asimetria,ks_train,ks_train_p,ks_inferencia,corr_inferencia,dif_mediana_inferencia
modelo,,,,,,,,,,,,,,
Ridge,0,0,0,50902.449,471964.230,177318.072,159365.832,71924.615,1.175,0.043,0.137,0.027,0.982,6315.195
ElasticNet,0,0,0,50969.523,472215.963,177192.442,159203.978,71502.126,1.174,0.041,0.161,0.027,0.983,6280.123
RandomForest,0,0,0,56821.040,477591.457,176176.310,155810.881,70123.338,1.358,0.053,0.031,0.017,0.993,4095.383
GradientBoosting,0,0,0,45703.472,525435.412,177541.695,156400.540,75293.797,1.451,0.048,0.070,0.000,1.000,0.000
HistGradientBoosting,0,0,0,50742.246,507499.257,177585.512,156742.711,75585.818,1.408,0.040,0.185,0.020,0.995,3368.905
MLP,0,0,0,52262.923,489101.883,178201.779,157834.141,75317.430,1.378,0.044,0.114,0.022,0.982,6359.427


## 7. Evaluación de coherencia

Un modelo se descarta si incumple alguna de estas reglas sobre sus predicciones en `test.csv`:

- Sin valores vacíos ni negativos.
- Todas las predicciones dentro del rango de precios de `train.csv`.
- Correlación contra la inferencia de al menos 0.95.
- Distancia KS contra `train.csv` de 0.10 como máximo.

In [7]:
MIN_CORR_INFERENCIA = 0.95
MAX_KS_TRAIN = 0.10


def coherence_issues(row):
    """Devuelve los motivos por los que un modelo no es coherente."""
    issues = []
    if row["vacios"] > 0:
        issues.append(f"{row['vacios']} predicciones vacías")
    if row["negativos"] > 0:
        issues.append(f"{row['negativos']} predicciones no positivas")
    if row["fuera_de_rango"] > 0:
        issues.append(f"{row['fuera_de_rango']} predicciones fuera del rango de train")
    if row["corr_inferencia"] < MIN_CORR_INFERENCIA:
        issues.append(f"correlación contra inferencia de {row['corr_inferencia']:.3f}")
    if row["ks_train"] > MAX_KS_TRAIN:
        issues.append(f"distancia KS contra train de {row['ks_train']:.3f}")
    return issues


coherence = {name: coherence_issues(row) for name, row in test_summary.iterrows()}

pd.DataFrame(
    {
        "estado": ["Descartado" if issues else "Coherente" for issues in coherence.values()],
        "motivos": ["; ".join(issues) or "-" for issues in coherence.values()],
    },
    index=coherence,
)

,estado,motivos
Ridge,Coherente,-
ElasticNet,Coherente,-
RandomForest,Coherente,-
GradientBoosting,Coherente,-
HistGradientBoosting,Coherente,-
MLP,Coherente,-


## 8. Curvas de aprendizaje

Error de entrenamiento y de validación cruzada por tamaño de muestra, con el preprocesamiento reajustado en cada fold.

In [8]:
learning_curves = {}

for name, search in searches.items():
    sizes, train_scores, cv_scores = learning_curve(
        clone(search.best_estimator_),
        X_train,
        y_train,
        cv=cv,
        train_sizes=TRAIN_SIZES,
        scoring=SCORING,
        n_jobs=-1,
        error_score="raise",
    )
    learning_curves[name] = {
        "sizes": sizes.tolist(),
        "train": (-train_scores).mean(axis=1).tolist(),
        "cv": (-cv_scores).mean(axis=1).tolist(),
        "cv_std": cv_scores.std(axis=1).tolist(),
    }

print(f"Curvas calculadas: {len(learning_curves)}")

Curvas calculadas: 6


## 9. Exportación de resultados

Se guardan en `reports/model_comparison.json` las métricas, los folds, todas las configuraciones evaluadas, las curvas de aprendizaje, las predicciones de validación, las densidades y las diferencias contra la inferencia. Las densidades se evalúan en una malla común de 0 a 800,000 dólares.

In [9]:
OUTPUT_PATH = PROJ_ROOT / "reports" / "model_comparison.json"
PRICE_GRID = np.linspace(0, 800_000, 161)
DIFF_EDGES = np.arange(-100_000, 105_000, 5_000)


def density(values):
    """Densidad por kernel en la malla común, en unidades de 1e-6 por dólar."""
    return (stats.gaussian_kde(values)(PRICE_GRID) * 1e6).round(4).tolist()


def clean_params(params):
    """Quita los prefijos del pipeline en los nombres de los hiperparámetros."""
    return {
        key.split("__")[-1]: str(value) if isinstance(value, tuple) else value
        for key, value in params.items()
    }


def export_model(name):
    search = searches[name]
    results = search.cv_results_
    best = search.best_index_
    configs = [
        {
            "params": clean_params(results["params"][i]),
            "cv": -results["mean_test_score"][i],
            "cv_std": results["std_test_score"][i],
            "train": -results["mean_train_score"][i],
        }
        for i in range(len(results["params"]))
    ]
    diff = test_predictions[name] - inference
    return {
        "best_params": clean_params(search.best_params_),
        "metrics": metrics.set_index("modelo").loc[name].to_dict(),
        "folds": [-results[f"split{i}_test_score"][best] for i in range(N_FOLDS)],
        "configs": sorted(configs, key=lambda item: item["cv"]),
        "learning": learning_curves[name],
        "pred_valid": valid_predictions[name].round().tolist(),
        "test_density": density(test_predictions[name]),
        "test_summary": test_summary.loc[name].to_dict(),
        "diff_hist": np.histogram(np.clip(diff, DIFF_EDGES[0], DIFF_EDGES[-1]), bins=DIFF_EDGES)[0].tolist(),
        "issues": coherence[name],
    }


comparison = {
    "winner": config["WINNER"]["NAME"],
    "n_folds": N_FOLDS,
    "target_range": float(target_range),
    "thresholds": {"min_corr_inferencia": MIN_CORR_INFERENCIA, "max_ks_train": MAX_KS_TRAIN},
    "price_grid": PRICE_GRID.tolist(),
    "diff_edges": DIFF_EDGES.tolist(),
    "train_density": density(train_prices),
    "inference_density": density(inference),
    "train_summary": {
        "minimo": float(train_prices.min()),
        "maximo": float(train_prices.max()),
        "media": float(train_prices.mean()),
        "mediana": float(np.median(train_prices)),
    },
    "y_valid": y_valid_real.tolist(),
    "models": {name: export_model(name) for name in metrics["modelo"]},
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(comparison, f, ensure_ascii=False, default=float)

print(f"Resultados guardados en {OUTPUT_PATH}")

Resultados guardados en ..\reports\model_comparison.json
